In [ ]:
# Backpropagation vs Simple Predictive Coding

In [1]:
import torch
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# Load Boston housing
X, y = fetch_openml(name="boston", version=1, as_frame=False, return_X_y=True)

# Standardize
X = StandardScaler().fit_transform(X)
y = y.reshape(-1, 1)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)


In [3]:
# Start with Backpropagation

In [4]:
torch.manual_seed(0)

model = torch.nn.Sequential(
    torch.nn.Linear(13, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 1)
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = torch.nn.MSELoss()


In [5]:
for epoch in range(500):
    y_hat = model(X_train)
    loss = loss_fn(y_hat, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"[BP] Epoch {epoch}, Loss: {loss.item():.4f}")


[BP] Epoch 0, Loss: 602.0684
[BP] Epoch 100, Loss: 20.4941
[BP] Epoch 200, Loss: 9.4621
[BP] Epoch 300, Loss: 7.5011
[BP] Epoch 400, Loss: 6.5548


In [7]:
with torch.no_grad():
    for i in range(10):
        y_hat = model(X_test[i])
        print(y_hat, y_test[i])
    

tensor([22.0505]) tensor([22.6000])
tensor([28.3874]) tensor([50.])
tensor([23.9856]) tensor([23.])
tensor([11.5492]) tensor([8.3000])
tensor([18.7234]) tensor([21.2000])
tensor([18.4164]) tensor([19.9000])
tensor([24.3049]) tensor([20.6000])
tensor([21.0472]) tensor([18.7000])
tensor([18.3951]) tensor([16.1000])
tensor([14.3866]) tensor([18.6000])


In [8]:
with torch.no_grad():
    test_loss = loss_fn(model(X_test), y_test)
    print("Backprop test MSE:", test_loss.item())


Backprop test MSE: 20.21933937072754


# Predictive Coding

In predictive coding:
- Higher layers generate predictions of lower layers
- Information flows top → down

Predictive coding is defined with respect to a generative model.

$$x_{l} = f(W_{l+1} x_{l+1})+ϵ_{l}$$

Key points:
- Each layer predicts the layer below
- Nonlinearity is applied to the state, not the prediction error
- Errors are computed in the space of the state

Let's assume the generative model as a 3-Layer network:

$$ h = f(W_{1}x) + ϵ_{h} $$
$$ y = W_{2} f(h) + ϵ_{y} $$

Where:
- x = observed input (house features)
- h = latent (hidden) causes
- y = observed targets (hous prices)
- f = tanh nonlinearity
- ϵ = Gaussian noise

Predictive coding minimizes the sum of squared prediction errors.

## Inference

In predictive coding, we have latent neural activities (hidden states) h that are not observed.

The network defines a generative model:

$$h_{pred} = f(W_{1}x)$$
$$y_{pred} = W_{2} f(h)$$

We have prediction errors:
$$ϵ_{h} = h_{pred} - h$$
$$ϵ_{y} = y_{pred} - y$$

To minimize the total energy (sum of squared prediction errors):

$$F=ϵ_{h}^2 + ϵ_{y}^2$$

Inference is done by iterating through the generative model and optimizing h:

Because he generative model is a dynamical system we have to iterate several times (hyper paramerer) to get closer to a fix point for h:
- h must adjust iteratively to reduce errors.
- gradient decent in h-space with learning rate $η$
$$h \leftarrow h - \eta \frac{\partial F}{\partial h}$$

## Weight updates in predictive coding

The weights are updated to reduce the energy once the hidden states h have settled:

$$W_{1} \leftarrow W_{1} − \eta_{w}  \frac{\partial F}{\partial W_{1}}$$
$$W_{2} \leftarrow W_{2} − \eta_{w}  \frac{\partial F}{\partial W_{2}}$$


where $\eta_{w}$ is the weight learning rate.

## Compute $\frac{\partial F}{\partial W_{2}}$

$$F=ϵ_{h}^2 + ϵ_{y}^2 = (h_{pred} - h)^2 + (y_{pred} - h)^2$$

$$F=(h - \tanh(W_{1}x))^2 + (y - W_{2} \tanh(h))^2$$

$$\frac{\partial F}{\partial W_{2}} = \tanh(h)^{T}(y_{pred} − y) = \tanh(h)^{T} ϵ_{y} $$




## Compute $\frac{\partial F}{\partial W_{1}}$

$$\frac{\partial F}{\partial W_{1}} = -x^{T}(ϵ_{h} ⊙ (1−\tanh^{2}(h_{pred})) )$$

where $⊙$ element-wise multiplication 

In [15]:
# Load data
X, y = fetch_openml(name="boston", version=1, as_frame=False, return_X_y=True)

X = StandardScaler().fit_transform(X)
y = y.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

# Normalize targets
y_mean, y_std = y_train.mean(), y_train.std()
y_train = torch.tensor((y_train - y_mean) / y_std, dtype=torch.float32)
y_test  = torch.tensor((y_test  - y_mean) / y_std, dtype=torch.float32)


In [16]:
torch.manual_seed(0)

D_in, D_hidden, D_out = 13, 32, 1

W1 = torch.randn(D_in, D_hidden) * 0.1
W2 = torch.randn(D_hidden, D_out) * 0.1

lr_x = 0.1 #e-3    # inference step
lr_w = 1e-3    # learning rate
n_infer = 20


In [17]:
for epoch in range(100):
    total_loss = 0.0

    for x, y in zip(X_train, y_train):
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        # latent state
        h = torch.zeros(1, D_hidden, requires_grad=True)

        # ----- Inference -----
        for _ in range(n_infer):
            pred_h = torch.tanh(x @ W1)
            pred_y = torch.tanh(h) @ W2

            eps_h = h - pred_h
            eps_y = pred_y - y

            energy = (eps_h**2).sum() + (eps_y**2).sum()

            h.grad = None
            energy.backward()

            with torch.no_grad():
                h -= lr_x * h.grad

        # ----- Learning -----
        with torch.no_grad():
            W2 -= lr_w * torch.tanh(h).T @ eps_y
            W1 -= lr_w * x.T @ (eps_h * (1 - pred_h**2))

        total_loss += (eps_y**2).item()

    if epoch % 50 == 0:
        print(f"[PC-tanh] Epoch {epoch}, Loss: {total_loss / len(X_train):.4f}")


[PC-tanh] Epoch 0, Loss: 0.7547
[PC-tanh] Epoch 50, Loss: 0.1641


In [18]:
with torch.no_grad():
    h = X_test @ W1
    y_pred = h @ W2
    y_pred = (y_pred+y_mean)*y_std
    y = (y_test+y_mean)*y_std

    test_mse = ((y_pred - y)**2).mean()
    print("Predictive coding test MSE:", test_mse.item())


Predictive coding test MSE: 1476.3277587890625


In [19]:
with torch.no_grad():
    for i in range(10):
        x = X_test[i]
        h = x @ W1
        y_pred = h @ W2
        y =  y_test[i]
        print((y+y_mean)*y_std, (y_pred+ y_mean)*y_std)
        

tensor([208.6355]) tensor([209.8529])
tensor([236.0356]) tensor([266.8424])
tensor([209.0356]) tensor([109.1243])
tensor([194.3355]) tensor([267.8290])
tensor([207.2355]) tensor([209.1352])
tensor([205.9355]) tensor([219.2628])
tensor([206.6355]) tensor([263.4854])
tensor([204.7355]) tensor([209.1774])
tensor([202.1355]) tensor([214.0050])
tensor([204.6355]) tensor([202.5657])
